This notebook will contain no serious model training. Its purpose is to lock down the experiment so we do not accidently introduce leakage or make methodological decisions halfway through modelling.

We complete the following steps:
| Step | What you need to decide/do  | Output                                                        |
| ---- | --------------------------- | ------------------------------------------------------------- |
| 1    | Define prediction timing    | Clear statement of what information exists at prediction time |
| 2    | Define targets              | Regression + classification                                   |
| 3    | Define modelling features   | Final initial `X` columns                                     |
| 4    | Create chronological splits | Train / validation / test                                     |
| 5    | Calculate trivial baselines | Numbers ML must beat                                          |


# Load Data

In [7]:
TABLE = "price_features"
CALENDAR = "no_calendar"

In [8]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [9]:
import pandas as pd
import numpy as np
import duckdb

from src.config import (
    DATABASE_PATH,
    ROLLING_WINDOWS,
    LAGGED_WINDOWS,
    MA_WINDOWS,
    TICKERS,
    START_DATE,
    END_DATE
)

from src.features import (
    build_price_feature_columns
)

CALENDAR = "no_calendar"


def pull_calendar_table(table_name: str, calendar: str) -> pd.DataFrame:

    print("[START] Connecting to database...")

    con = duckdb.connect(DATABASE_PATH)

    print(f"[START] Reading {table_name}")

    df = con.sql(f"""
    
        SELECT *
        FROM {table_name}

    """).df()

    print(f"[DONE] Read {table_name}")

    con.close()

    print(f"[START] Building {CALENDAR} required columns...")

    base_cols = [
        "ticker",
        "date",
        "open",
        "high",
        "low",
        "close",
        "adj_close",
        "volume"
    ]

    feature_columns = build_price_feature_columns(
        calendars = [CALENDAR],
        rolling_windows = ROLLING_WINDOWS,
        lagged_windows = LAGGED_WINDOWS,
        ma_windows = MA_WINDOWS
    )

    expected_columns = base_cols + list(feature_columns.keys())

    calendar_df = df[expected_columns].copy()

    print(f"[DONE] Filtered to {CALENDAR}")

    calendar_df["date"] = pd.to_datetime(calendar_df["date"])

    calendar_df = calendar_df.sort_values(["ticker", "date"])

    print(f"[DONE] Prepared and sorted table")

    return calendar_df

In [10]:
df = pull_calendar_table(
    table_name = TABLE,
    calendar = CALENDAR
)

[START] Connecting to database...
[START] Reading price_features
[DONE] Read price_features
[START] Building no_calendar required columns...
[DONE] Filtered to no_calendar
[DONE] Prepared and sorted table


In [11]:
df.columns

Index(['ticker', 'date', 'open', 'high', 'low', 'close', 'adj_close', 'volume',
       'daily_return_no_calendar', 'log_return_no_calendar',
       'cumulative_returns_no_calendar', 'rolling_7d_return_no_calendar',
       'rolling_30d_return_no_calendar', 'lag_1_return_no_calendar',
       'lag_5_return_no_calendar', 'moving_avg_20_no_calendar',
       'moving_avg_50_no_calendar', 'price_vs_ma20_no_calendar',
       'relative_volume_no_calendar', 'rolling_30d_volatility_no_calendar',
       'drawdown_no_calendar', 'target_next_day_return_no_calendar',
       'target_direction_no_calendar'],
      dtype='str')

In [12]:
table_count = 0
graph_count = 0
matrix_count = 0
figure_count = 0

# Define Prediction Timing

We make predictions based on data that is only available at the end of the trading day and make a prediction for the next day.

In [35]:
df_1 = df.copy()

ticker = "ticker"
daily_return = "daily_return_no_calendar" 
log_return = "log_return_no_calendar"
target_return = "target_next_day_return_no_calendar"
target_direction = "target_direction_no_calendar" 

columns =[ticker, daily_return, log_return, target_return, target_direction]

df_1 = df_1[columns]

summary_rows = []

for asset, group in df_1.groupby(ticker):

    expected_simple_direction = (group[daily_return].shift(-1) > 0).astype(int)
    expected_log_direction = (group[log_return].shift(-1) > 0).astype(int)

    summary_row = pd.DataFrame({
        "Ticker": asset,

        "Simple Return Match": [
            (group[daily_return].shift(-1).iloc[:-1] == group[target_return].iloc[:-1]).all()
        ],

        "Log Return Match": [
            (group[log_return].shift(-1).iloc[:-1] == group[target_return].iloc[:-1]).all()
        ],

        "Simple Direction Match": [
            (group[target_direction].iloc[:-1] == expected_simple_direction.iloc[:-1]).all()
        ],

        "Log Direction Match": [
            (group[target_direction].iloc[:-1] == expected_log_direction.iloc[:-1]).all()
        ]
    })

    summary_rows.append(summary_row)

summary_table = pd.concat(summary_rows)

display(summary_table.style.hide(axis="index"))


Ticker,Simple Return Match,Log Return Match,Simple Direction Match,Log Direction Match
GLD,True,False,True,True
MU,True,False,True,True
NKE,True,False,True,True
RPI.L,True,False,True,True
SNDK,True,False,True,True
SPY,True,False,True,True
TLT,True,False,True,True


Table shows the correct alignment of targets and input rows.

# Define Targets

We will experiment with both regression - predicting target next day returns, and classification - predicting target next day direction.

We will treat them as separate experiements building and testing separate models for each. We may evantually conclude that predicting one target works better compared to predicting the other.

# Define Modelling Features

We will establish a baseline based on a delibirately reasonable feature set.

```
daily_return
lag_1_return
lag_5_return
rolling_7d_return
rolling_30d_return
price_vs_ma20
moving_avg_20
moving_avg_50
rolling_30d_volatility
drawdown
relative_volume
```

It is important here that we do not include target vairables, future values, duplicate versions of the same information (e.g., simple and log daily return) unless we want to delibirately test them, other calendar versions.

We will then evaluate whether feature reduction, addition or alternative combinations will help the model prediction.

In [54]:
df_2 = df.copy()

all_cols = df_2.columns

generated_cols = [
    col for col in all_cols
    if col.endswith("_no_calendar")
]

baseline_features = ['daily_return_no_calendar', 'lag_1_return_no_calendar',
       'lag_5_return_no_calendar', 'rolling_7d_return_no_calendar',
       'rolling_30d_return_no_calendar', 'moving_avg_20_no_calendar',
       'moving_avg_50_no_calendar', 'price_vs_ma20_no_calendar',
       'relative_volume_no_calendar', 'rolling_30d_volatility_no_calendar',
       'drawdown_no_calendar']

target_features = ['target_next_day_return_no_calendar', 'target_direction_no_calendar']

additional_features = [
    col for col in generated_cols
    if col not in baseline_features and col not in target_features
]

base_cols = [
    col for col in all_cols
    if col not in generated_cols
]

all_sets = baseline_features + target_features + additional_features + base_cols

set_check = (
    len(all_sets) == len(set(all_sets))
    and set(all_sets) == set(all_cols)
)

print("Baseline features:")
print(baseline_features)

print("\n")
print("Target features:")
print(target_features)

print("\n")
print("Additional features:")
print(additional_features)

print("\n")
print("Original columns:")
print(base_cols)

print("\n")
print("Set Check:")
print(set_check)



Baseline features:
['daily_return_no_calendar', 'lag_1_return_no_calendar', 'lag_5_return_no_calendar', 'rolling_7d_return_no_calendar', 'rolling_30d_return_no_calendar', 'moving_avg_20_no_calendar', 'moving_avg_50_no_calendar', 'price_vs_ma20_no_calendar', 'relative_volume_no_calendar', 'rolling_30d_volatility_no_calendar', 'drawdown_no_calendar']


Target features:
['target_next_day_return_no_calendar', 'target_direction_no_calendar']


Additional features:
['log_return_no_calendar', 'cumulative_returns_no_calendar']


Original columns:
['ticker', 'date', 'open', 'high', 'low', 'close', 'adj_close', 'volume']


Set Check:
True


# Create Chronological Splits

We should never randomly split time series data - the model is always going to be trained on past data and evaluated on future observations. The requirement is that train < validation < test.

It is important that we do not force the same number of observations within each set. For later IPO assets like SNDK and RPI.L we shoudl split by percent within each ticker. When building a pooled model we can determine how to deal with differing start dates.

One thing that is worrying is the recent market trends and changes that have diverted from past behaviours.

In [57]:
# Split in proportions by ticker

df_3 = df.copy()
ticker = "ticker"

train_sets = []
validation_sets = []
test_sets = []

for asset, group in df_3.groupby(ticker):

    group = group.sort_values("date").reset_index(drop=True)

    n = len(group)

    train_end = int(n * 0.7)
    validation_end = int(n * 0.85)

    train_sets.append(group.iloc[:train_end])
    validation_sets.append(group.iloc[train_end:validation_end])
    test_sets.append(group.iloc[validation_end:])

train_ticker = pd.concat(train_sets, ignore_index=True)
validation_ticker = pd.concat(validation_sets, ignore_index=True)
test_ticker = pd.concat(test_sets, ignore_index=True)

data_split_summary = pd.concat([
    train_ticker.assign(split="Train"),
    validation_ticker.assign(split="Validation"),
    test_ticker.assign(split="Test")
])

data_split_summary = (
    data_split_summary
    .groupby([ticker, "split"])
    .agg(
        rows=("date", "size"),
        start_date=("date","min"),
        end_date=("date","max")
    ).reset_index()
)

display(data_split_summary.style.hide(axis="index"))

ticker,split,rows,start_date,end_date
GLD,Test,324,2025-04-16 00:00:00,2026-07-31 00:00:00
GLD,Train,1509,2018-01-02 00:00:00,2023-12-29 00:00:00
GLD,Validation,323,2024-01-02 00:00:00,2025-04-15 00:00:00
MU,Test,324,2025-04-16 00:00:00,2026-07-31 00:00:00
MU,Train,1509,2018-01-02 00:00:00,2023-12-29 00:00:00
MU,Validation,323,2024-01-02 00:00:00,2025-04-15 00:00:00
NKE,Test,324,2025-04-16 00:00:00,2026-07-31 00:00:00
NKE,Train,1509,2018-01-02 00:00:00,2023-12-29 00:00:00
NKE,Validation,323,2024-01-02 00:00:00,2025-04-15 00:00:00
RPI.L,Test,82,2026-04-07 00:00:00,2026-07-31 00:00:00


We will train models based on individual ticker splits using the 70:15:15 train:validation:test split. I am slightly worried since the test set captures all recent data and so is likely to not match the behvaiour seen in the past.

In [61]:
# Pooled chronological splits

df_4 = df.copy()
date = "date"

unique_dates = np.sort(df_4[date].unique())

train_end = int(len(unique_dates) * 0.7)
validation_end = int(len(unique_dates) * 0.85)

train_end_date = unique_dates[train_end]
validation_end_date = unique_dates[validation_end]

pooled_train = df_4[df_4[date] < train_end_date].copy()
pooled_validation = df_4[df_4[date] < validation_end_date].copy()
pooled_test = df_4[df_4[date] >= validation_end_date].copy()

pooled_split_summary = pd.DataFrame({
    "split": ["Train", "Validation", "Test"],
    "rows": [
        len(pooled_train),
        len(pooled_validation),
        len(pooled_test)
    ],
    "start_date": [
        pooled_train["date"].min(),
        pooled_validation["date"].min(),
        pooled_test["date"].min()
    ],
    "end_date": [
        pooled_train["date"].max(),
        pooled_validation["date"].max(),
        pooled_test["date"].max()
    ]
})

display(pooled_split_summary.style.hide(axis="index"))

split,rows,start_date,end_date
Train,7595,2018-01-02 00:00:00,2024-01-16 00:00:00
Validation,9461,2018-01-02 00:00:00,2025-04-24 00:00:00
Test,2228,2025-04-25 00:00:00,2026-07-31 00:00:00


For now we used manual chronological slicing for the train/validation/test sets but we will potentially use TimeSeriesSplit later if we decide to do cross validation or hyperparameter tuning.

With respect to shifting regimes, especially in more recent times, we may perform a walk-forward / expanding window validation alongside the final holdout test. This may look something like:

```
Fold 1
TRAIN:  2018 ───────── 2021
VAL:                       2022

Fold 2
TRAIN:  2018 ───────────────── 2022
VAL:                              2023

Fold 3
TRAIN:  2018 ─────────────────────── 2023
VAL:                                  2024

Fold 4
TRAIN:  2018 ───────────────────────────── 2024
VAL:                                        2025

FINAL TEST:                                  2025/26 ──>
```

However, due to the use of our features a shift in regimes doesn't necessarily mean that the older data is useless. For example, training using features means that when moving average is below a certtain threshold while volume is above a threshold we are more likely to see a price increase.

# Calculate Trivial Baselines

Regression baselines:
- always predicting 0.
- always predicting the training sets mean next day return.

Regression evaluated with:
- MAE
- RMSE
- R^2

Classification baselines:
- always predict the majority training class.

Classification evaluated with:
- accuracy
- precision
- recall
- F1

We apply these baselines to tickers individually then on the pooled set.

When coming to the actual modelling in later notebooks we will have an additional column that indicates the difference.

We will first create baselines on the validation sets and use those as comparisons until final evaluation when we will recreate the baselines on the test sets.

In [ ]:
# Always predicting zeros

from sklearn.metrics import (mean_absolute_error,
                            root_mean_squared_error,
                            r2_score)

val_set1 = validation_ticker.copy()
ticker = "ticker"
target = "target_next_day_return_no_calendar"

zero_results = []

for asset, group in val_set1.groupby(ticker):

    val_length = len(group)

    y_pred = np.zeros(val_length)

    y_actual = group[target]

    mae = mean_absolute_error(y_actual, y_pred)

    rmse = root_mean_squared_error(y_actual, y_pred)

    r2_s = r2_score(y_actual, y_pred)

    zero_result = pd.DataFrame({
        "Ticker": [asset],
        "MAE": [mae],
        "RMSE": [rmse],
        "R^2": [r2_s]
    })

    zero_results.append(zero_result)

results = pd.concat(zero_results)

display(results.style.hide(axis="index"))


Ticker,MAE,RMSE,R^2
GLD,0.759164,0.991006,-2.433937
MU,2.670843,3.752859,-0.002626
NKE,1.392219,2.321591,-0.590990
RPI.L,4.156743,8.039907,-0.785858
SNDK,5.287747,6.439544,-6.725984
SPY,0.714254,1.143330,-0.150091
TLT,0.716577,0.901096,-0.022431
